# 빅데이터프로그래밍 1차 해커톤 최종 제출 #27

계명대학교 경영정보학과 남종욱 · 2026년 11월 25일

PyTorch 기본 레이어로 4블록 잔차 CNN을 구현해 12개 클래스를 분류했습니다.
교수님 평가 사이트 제출 #27은 60장 중 37장, **61.67%**, 독립 생성 18장 중 14장, **77.78%**를 기록했습니다.
이 노트북의 기본 실행에서는 제출 당시의 가중치로 데이터를 복원하고 평가합니다.
검증으로 가중치를 고른 학습 절차는 선택 실행으로 제공합니다. 코드 아래에는 보관된 실제 성능 기록도 포함했습니다.

처음에는 Kaggle과 Hugging Face 데이터를 가져오고 이미지 중복을 검사합니다. Colab GPU 런타임에서 실행하면 시간이 줄어듭니다.
[Kaggle V1](https://www.kaggle.com/datasets/mostafaabla/garbage-classification) · [Kaggle V2 v12](https://www.kaggle.com/datasets/sumn2u/garbage-classification-v2/versions/12) · [TrashBusters](https://huggingface.co/datasets/TrashBusters/combined)

## 1 Kaggle 데이터와 중복 정리

In [ ]:
%pip -q install kagglehub ImageHash huggingface_hub pyarrow

In [ ]:
import kagglehub
from pathlib import Path

# 지정 데이터셋을 가져오고 저장 경로 확인
data_path = Path(kagglehub.dataset_download("mostafaabla/garbage-classification"))
print("데이터 경로:", data_path)

# 폴더 구조 확인
print("내부 항목:", sorted(p.name for p in data_path.iterdir()))
# 클래스별 폴더가 있는 경로
image_dir = data_path / "garbage_classification"

# 클래스별 이미지 수 집계
extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
class_counts = {
    folder.name: sum(
        p.is_file() and p.suffix.lower() in extensions
        for p in folder.rglob("*")
    )
    for folder in sorted(image_dir.iterdir())
    if folder.is_dir()
}

for name, count in class_counts.items():
    print(f"{name}: {count:,}장")

print(f"\n클래스 수: {len(class_counts)}개")
print(f"전체 이미지: {sum(class_counts.values()):,}장")
from PIL import Image

In [ ]:
import pandas as pd

records, bad_files = [], []

# 모든 이미지를 읽어 크기·색상 모드·손상 여부 확인
for name in class_counts:
    for path in sorted((image_dir / name).rglob("*")):
        if not path.is_file() or path.suffix.lower() not in extensions:
            continue
        try:
            with Image.open(path) as img:
                img.load()  # 실제 픽셀까지 읽어 오류 확인
                records.append({
                    "path": str(path), "class": name,
                    "width": img.width, "height": img.height, "mode": img.mode
                })
        except (OSError, ValueError) as error:
            bad_files.append((str(path), str(error)))

images_df = pd.DataFrame(records)

print("정상 이미지:", len(images_df))
print("손상 이미지:", len(bad_files))
print("\n이미지 크기 요약:\n", images_df[["width", "height"]].describe().round(1))
print("\n색상 모드별 수량:\n", images_df["mode"].value_counts())
print("\n손상 파일 예시:", bad_files[:5])
import hashlib

# 파일 내용으로 해시 생성: 내용이 같으면 같은 해시가 나옵니다.
images_df["hash"] = images_df["path"].apply(
    lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
)

# 중복 그룹의 이미지 수와 클래스 확인
duplicates = images_df[images_df.duplicated("hash", keep=False)]
duplicate_groups = duplicates.groupby("hash").agg(
    count=("path", "size"),
    classes=("class", lambda x: sorted(set(x)))
)

print("중복 그룹 수:", len(duplicate_groups))
print("중복 여분 이미지 수:", images_df.duplicated("hash").sum())
print("클래스가 서로 다른 중복 그룹 수:",
      duplicate_groups["classes"].apply(len).gt(1).sum())
print("\n중복 그룹 예시:\n", duplicate_groups.head(10).to_string(index=False))
conflict_hashes = duplicate_groups.index[duplicate_groups["classes"].apply(len).gt(1)]
# 레이블이 충돌하는 이미지 제외: 제외 목록은 보고서용으로 보관
conflict_rows = images_df[images_df["hash"].isin(conflict_hashes)].copy()
clean_df = images_df[~images_df["hash"].isin(conflict_hashes)].copy()

# 같은 클래스의 동일 파일은 한 장만 유지
duplicate_rows = clean_df[clean_df.duplicated("hash")].copy()
clean_df = clean_df.drop_duplicates("hash").reset_index(drop=True)

print("레이블 충돌로 제외:", len(conflict_rows))
print("동일 클래스 중복으로 제외:", len(duplicate_rows))
print("최종 이미지 수:", len(clean_df))
print("\n클래스별 수량:\n", clean_df["class"].value_counts().sort_index())

In [ ]:
#@title V1 분할 목록
import base64, io, json, zlib

EMBEDDED_SPLIT_B64 = """
...
"""

EXPECTED_DATA_SHA256 = "b6b506e1be90c15e8b3d3e72460f30fa2425402eef7912118f5245ee6434f271"

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 클래스 이름을 정수 레이블로 변환: 모든 데이터에 같은 매핑 사용
classes = sorted(clean_df["class"].unique())
class_to_idx = {name: idx for idx, name in enumerate(classes)}

# 크기를 통일하고 픽셀 값을 [0, 1]에서 [-1, 1]로 변환
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3)
])

class GarbageDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        with Image.open(row["path"]) as img:
            image = self.transform(img.convert("RGB"))
        return image, class_to_idx[row["class"]]


from torch import nn

set_signature = sorted(
    label + "/" + Path(path).name + "\t" + digest
    for label, path, digest in zip(clean_df["class"], clean_df["path"], clean_df["hash"])
)
assert hashlib.sha256("\n".join(set_signature).encode()).hexdigest() == EXPECTED_DATA_SHA256, (
    "V1 이미지가 원래 학습에 사용한 파일과 다릅니다."
)
original_split = pd.DataFrame(
    json.loads(zlib.decompress(base64.b64decode(EMBEDDED_SPLIT_B64))),
    columns=["class", "filename", "split"]
)
original_split["path"] = [
    str(image_dir / name / filename)
    for name, filename in zip(original_split["class"], original_split["filename"])
]
original_train, original_val, original_test = [
    original_split[original_split["split"] == part].copy() for part in ("train", "val", "test")
]
assert [len(original_train), len(original_val), len(original_test)] == [12394, 1549, 1550]

In [ ]:
eda_counts = pd.crosstab(original_split["class"], original_split["split"])[["train", "val", "test"]]
ax = eda_counts.plot.bar(stacked=True, figsize=(10, 3.5), color=["#41665A", "#80A899", "#C3D8CD"])
ax.set(xlabel="Class", ylabel="Images", title="Kaggle V1 after removing exact duplicates")
display(eda_counts)

In [ ]:
V2_HANDLE = "sumn2u/garbage-classification-v2/versions/12"
EXTRA_TARGETS = {
    "biological": 1000, "plastic": 1000, "trash": 800,
    "metal": 500, "shoes": 300, "battery": 200
}
ADDITIONAL_LR = 0.0003
ADDITIONAL_EPOCHS = 10
SEED = 42
PHASH_DISTANCE = 4  # 유사도 후보 기준이며 라벨을 결정하는 값이 아님

assert set(EXTRA_TARGETS).issubset(class_to_idx) and not any("glass" in c for c in EXTRA_TARGETS)
assert PHASH_DISTANCE == 4, "아래 pHash 인덱스는 거리 4를 기준으로 구성되어 있습니다."

v2_path = Path(kagglehub.dataset_download(V2_HANDLE))
original_dirs = [v2_path / "original"] if (v2_path / "original").is_dir() else list(v2_path.rglob("original"))
assert len(original_dirs) == 1, "V2의 original 폴더를 하나로 식별할 수 없습니다. 경로를 확인해 주세요."
v2_image_dir = original_dirs[0]
inventory = []
for folder in sorted(v2_image_dir.iterdir()):
    if folder.is_dir():
        paths = sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in extensions)
        inventory.extend({"path": str(p), "class": folder.name} for p in paths)
v2_inventory = pd.DataFrame(inventory)
assert not v2_inventory.empty, "V2 original 폴더에 이미지가 없습니다."
print("V2 실제 클래스별 수량:\n", v2_inventory["class"].value_counts().sort_index())
print("전체 V2 이미지:", len(v2_inventory))
v2_candidates = v2_inventory[v2_inventory["class"].isin(EXTRA_TARGETS)].reset_index(drop=True)
assert not v2_candidates.empty, "추가 대상 클래스의 폴더 이름을 확인해 주세요."

In [ ]:
import imagehash
from PIL import ImageOps
from collections import defaultdict

def image_fingerprint(path, file_hash=None):
    with Image.open(path) as img:
        rgb = ImageOps.exif_transpose(img).convert("RGB")
        pixels = hashlib.sha256(str(rgb.size).encode() + b"|" + rgb.tobytes()).hexdigest()
        perceptual = str(imagehash.phash(rgb, hash_size=8))
    return {
        "sha256": file_hash or hashlib.sha256(Path(path).read_bytes()).hexdigest(),
        "pixel_hash": pixels, "phash": perceptual
    }

# 64비트를 5구간으로 나누면 4비트 이내 차이는 적어도 한 구간이 일치
def hash_keys(value):
    return [(band, (value >> (13 * band)) & ((1 << min(13, 64 - 13 * band)) - 1))
            for band in range(5)]

def add_hash(index, hashes, value, row_id):
    hashes[row_id] = value
    for key in hash_keys(value):
        index[key].add(row_id)

def nearby_hashes(index, hashes, value):
    possible = set()
    for key in hash_keys(value):
        possible.update(index.get(key, ()))
    return [row_id for row_id in possible
            if (hashes[row_id] ^ value).bit_count() <= PHASH_DISTANCE]

# 기존 원본 전체의 해시: test와 겹친 외부 이미지도 학습에 들어가지 않게 함
base_sha = set(images_df["hash"])
base_pixels = set()
base_index, base_hashes = defaultdict(set), {}
for row_id, row in images_df.reset_index(drop=True).iterrows():
    fingerprint = image_fingerprint(row["path"], row["hash"])
    base_pixels.add(fingerprint["pixel_hash"])
    add_hash(base_index, base_hashes, int(fingerprint["phash"], 16), row_id)
    if (row_id + 1) % 2000 == 0:
        print(f"기존 이미지 해시 확인: {row_id + 1:,}/{len(images_df):,}")

audit_rows = []
for row_id, row in v2_candidates.iterrows():
    record = {"path": row["path"], "class": row["class"], "status": "candidate"}
    try:
        record.update(image_fingerprint(row["path"]))
    except (OSError, ValueError) as error:
        record.update(status="unreadable", error=str(error))
    audit_rows.append(record)
    if (row_id + 1) % 500 == 0:
        print(f"V2 이미지 해시 확인: {row_id + 1:,}/{len(v2_candidates):,}")
assert any(row["status"] == "candidate" for row in audit_rows), "읽을 수 있는 V2 추가 대상 이미지가 없습니다."
v2_audit = pd.DataFrame(audit_rows)

# 먼저 V2 내부의 동일 이미지·다른 라벨 충돌을 모두 제외
readable_mask = v2_audit["status"] == "candidate"
readable_rows = v2_audit[readable_mask]
for key in ["sha256", "pixel_hash"]:
    valid = readable_rows
    conflict_keys = valid.groupby(key)["class"].nunique()
    conflict_keys = set(conflict_keys[conflict_keys > 1].index)
    mask = readable_mask & v2_audit[key].isin(conflict_keys)
    v2_audit.loc[mask, "status"] = "label_conflict"

for row_id, row in v2_audit[v2_audit["status"] == "candidate"].iterrows():
    if row["sha256"] in base_sha or row["pixel_hash"] in base_pixels:
        v2_audit.loc[row_id, "status"] = "duplicate_of_original"
for key in ["sha256", "pixel_hash"]:
    valid = v2_audit[v2_audit["status"] == "candidate"]
    redundant = valid.index[valid.duplicated(key, keep="first")]
    v2_audit.loc[redundant, "status"] = "duplicate_within_v2"

# 유사 후보는 원본과의 비교, V2 상호 비교 모두 보류
extra_index, extra_hashes = defaultdict(set), {}
review_ids = set()
for row_id, row in v2_audit[v2_audit["status"] == "candidate"].iterrows():
    value = int(row["phash"], 16)
    if nearby_hashes(base_index, base_hashes, value):
        review_ids.add(row_id)
    neighbors = nearby_hashes(extra_index, extra_hashes, value)
    if neighbors:
        review_ids.add(row_id)
        review_ids.update(neighbors)
    add_hash(extra_index, extra_hashes, value, row_id)
if review_ids:
    v2_audit.loc[sorted(review_ids), "status"] = "near_duplicate_review"
novel_v2 = v2_audit[v2_audit["status"] == "candidate"].copy()
print("제외·보류 내역:\n", pd.crosstab(v2_audit["class"], v2_audit["status"]))
print("신규 후보 수:\n", novel_v2["class"].value_counts().sort_index())
assert not novel_v2.empty, "중복·유사 후보를 제외한 신규 이미지가 없습니다. 추가 학습을 진행하지 않습니다."

In [ ]:
EXPECTED_V2_SPLIT_SHA256 = "e1d622a7cd372e0f5e609bfb7e40ce7f40d0c58fd5a694628c6d57a508b658a1"

parts, selection_rows = [], []
for name, target in EXTRA_TARGETS.items():
    group = novel_v2[novel_v2["class"] == name]
    count = min(len(group), target)
    if count < 10:
        selection_rows.append({"class": name, "available": len(group), "selected": 0,
                               "train": 0, "val": 0, "test": 0})
        continue
    chosen = group.sample(n=count, random_state=SEED).reset_index(drop=True)
    holdout_count = max(1, round(count * 0.1))
    chosen["split"] = (["test"] * holdout_count + ["val"] * holdout_count
                       + ["train"] * (count - 2 * holdout_count))
    parts.append(chosen)
    selection_rows.append({"class": name, "available": len(group), "selected": count,
                           "train": count - 2 * holdout_count,
                           "val": holdout_count, "test": holdout_count})
assert parts, "분할할 수 있는 신규 데이터가 없습니다. 선별 결과를 확인해 주세요."
extra_split = pd.concat(parts, ignore_index=True)
extra_split["source"] = "sumn2u/garbage-classification-v2"
extra_split["source_version"] = 12
extra_split["original_label"] = extra_split["class"]
# pHash는 64비트 정밀도 보존을 위해 16진수 문자열 그대로 저장
selection_df = pd.DataFrame(selection_rows)

extra_train = extra_split[extra_split["split"] == "train"].copy()
extra_val = extra_split[extra_split["split"] == "val"].copy()
extra_test = extra_split[extra_split["split"] == "test"].copy()
combined_train = pd.concat([original_train, extra_train], ignore_index=True)
combined_val = pd.concat([original_val, extra_val], ignore_index=True)

# 분할 경로의 중복 확인: 이미지 내용 중복은 앞 셀에서 검사
frames = [combined_train, combined_val, original_test, extra_test]
path_sets = [set(frame["path"]) for frame in frames]
assert all(path_sets[i].isdisjoint(path_sets[j]) for i in range(4) for j in range(i + 1, 4))
assert len(original_test) == 1550 and len(original_val) == 1549
print("V2 선택·분할 수량:\n", selection_df.to_string(index=False))
print(f"학습 {len(combined_train):,} / 검증 {len(combined_val):,}")
print(f"기존 시험 {len(original_test):,} / V2 holdout {len(extra_test):,}")
print("추가하지 않은 클래스도 원래 train에서 계속 학습합니다:", classes)
assert len(v2_inventory) == 12259 and len(v2_candidates) == 5884
expected_v2 = {
    "biological": (48, 6, 6), "plastic": (772, 96, 96),
    "trash": (178, 22, 22), "metal": (255, 32, 32),
}
actual_v2 = pd.crosstab(extra_split["class"], extra_split["split"])[["train", "val", "test"]]
assert {name: tuple(row) for name, row in actual_v2.iterrows()} == expected_v2, (
    "V2 분할 수량이 실험 기록과 다릅니다."
)

# 원래 Colab CSV에는 절대 경로가 포함되어 있으므로 당시 경로로도 검사합니다.
paths = extra_split["path"].map(lambda path: Path(path).relative_to(v2_image_dir).as_posix())
colab_roots = (
    v2_image_dir.as_posix(), "/kaggle/input/garbage-classification-v2/original",
    "/kaggle/input/datasets/sumn2u/garbage-classification-v2/original",
    "/root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/12/original",
)
for root in dict.fromkeys(colab_roots):
    csv = extra_split.assign(path=root + "/" + paths).to_csv(index=False, lineterminator="\n")
    if hashlib.sha256(csv.encode()).hexdigest() == EXPECTED_V2_SPLIT_SHA256:
        print("V2 분할이 당시 Colab 분할과 일치합니다.")
        break
else:
    print("V2 분할을 당시 정렬과 seed로 재구성했습니다. 저장된 분할 해시는 원래 절대 경로를 포함합니다.")

## 2 모델 정의와 학습된 가중치 불러오기

각 블록에 3×3 합성곱 두 개와 잔차 연결을 사용합니다. 채널 수는 32 → 64 → 128 → 192이고,
전역 평균 풀링과 선형 분류기가 12개 클래스 점수를 계산합니다. 파라미터는 879,244개입니다.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.shortcut = nn.Identity()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        self.activation = nn.ReLU()
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        return self.pool(self.activation(self.convs(x) + self.shortcut(x)))


class ResidualCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            ResidualBlock(3, 32),
            ResidualBlock(32, 64),
            ResidualBlock(64, 128),
            ResidualBlock(128, 192),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(192, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
#@title #27 61.67% 가중치 (접힌 데이터 셀)
EMBEDDED_RESIDUAL_WEIGHTS = (
    "..."
)
EXPECTED_RESIDUAL_SHA256 = "882d802085f882f40576bf79d080db0800b1f943b0c18619b0c8dcd38ab3c184"

In [ ]:
residual_bytes = zlib.decompress(base64.b64decode(EMBEDDED_RESIDUAL_WEIGHTS))
if hashlib.sha256(residual_bytes).hexdigest() != EXPECTED_RESIDUAL_SHA256:
    raise ValueError("내장 잔차 CNN 가중치의 SHA-256이 일치하지 않습니다.")
final_model = ResidualCNN(len(classes)).to(device)
final_model.load_state_dict(torch.load(io.BytesIO(residual_bytes), map_location="cpu", weights_only=True))
final_model.eval()
print(f"평가 장치 {device} | 잔차 CNN #27 · 61.67%")

## #27의 기존 TrashBusters 분할 복원

기존 학습 3,421장, 검증 774장, 시험 777장을 같은 분할로 이어 씁니다. 분할표와 이미지 해시를
#27 학습 기록과 대조합니다. 새 실험 결과는 `residual_27_final` 폴더에 저장됩니다.

In [ ]:
import re
import pyarrow.parquet as pq
from huggingface_hub import snapshot_download

HF_SOURCE = "TrashBusters/combined"
HF_REVISION = "8c6e9226b01fc8a349750b69911b28d7d7c11443"
TARGET_CLASSES = ("paper", "trash", "plastic")
SPLITS = ("test", "validation", "train")  # 동일 사진이 여러 분할에 있으면 보류 시험을 우선 보존
assert set(TARGET_CLASSES).issubset(class_to_idx)

def allowed_hf_label(label, source, original):
    original = original.strip().casefold()
    if source in ("garbage_dataset", "trashnet"):
        return original == label
    if source == "household_garbage":
        if label == "paper":
            return original in {"newspaper", "office_paper", "magazines"}
        return label == "plastic" and original.startswith("plastic_")
    if source == "tricascade":
        name = re.sub(r"^[a-z](?:_[a-z])?_", "", original).replace("_", " ").strip()
        if label == "paper":
            return name in {"newspaper", "papers", "paper", "waste paper"}
        if label == "trash":
            return name in {"diaper", "diapers", "mask", "masks", "toothbrush", "toothbrushes"}
        return label == "plastic" and (name.startswith("plastic ") or name == "plastic")
    return False


def gather_hf_rows(files, cache):
    columns = ["image", "label_name", "source_dataset", "original_label", "image_context"]
    audit, saved_hashes = [], set()
    for split, paths in files.items():
        for parquet in paths:
            offset = 0
            print(f"TrashBusters 확인: {parquet.name}")
            for batch in pq.ParquetFile(parquet).iter_batches(batch_size=64, columns=columns):
                for i, row in enumerate(batch.to_pylist(), start=offset):
                    label = row["label_name"]
                    if label not in TARGET_CLASSES:
                        continue
                    source, original = row["source_dataset"] or "", row["original_label"] or ""
                    record = {
                        "id": f"{parquet.stem}:{i}", "split": split, "class": label,
                        "source_dataset": source, "original_label": original,
                        "image_context": row["image_context"], "status": "review_label",
                    }
                    audit.append(record)
                    if not allowed_hf_label(label, source, original):
                        continue
                    payload = (row["image"] or {}).get("bytes")
                    if not payload:
                        record["status"] = "unreadable"
                        continue
                    digest = hashlib.sha256(payload).hexdigest()
                    try:
                        with Image.open(io.BytesIO(payload)) as image:
                            rgb = ImageOps.exif_transpose(image).convert("RGB")
                            pixel_hash = hashlib.sha256(str(rgb.size).encode() + b"|" + rgb.tobytes()).hexdigest()
                            phash = str(imagehash.phash(rgb, hash_size=8))
                    except (OSError, ValueError, Image.DecompressionBombError):
                        record["status"] = "unreadable"
                        continue
                    # 파일명은 내용을 해시한 값으로 지정하고, 동일 바이트는 한 번만 저장합니다.
                    path = cache / (digest + ".img")
                    if digest not in saved_hashes:
                        path.write_bytes(payload)
                        saved_hashes.add(digest)
                    record.update(status="candidate", path=str(path), sha256=digest,
                                  pixel_hash=pixel_hash, phash=phash)
                offset += batch.num_rows
    return pd.DataFrame(audit)

In [ ]:
def deduplicate_hf(audit, used_v2):
    audit = audit.copy()
    v2_sha = set(used_v2["sha256"])
    v2_pixels = set(used_v2["pixel_hash"])
    v2_index, v2_hashes = defaultdict(set), {}
    for i, item in enumerate(used_v2["phash"]):
        add_hash(v2_index, v2_hashes, int(item, 16), i)

    valid = audit.loc[audit["status"].eq("candidate")]
    conflicts = set()
    for key in ("sha256", "pixel_hash"):
        label_counts = valid.groupby(key)["class"].nunique()
        conflicts.update(valid.index[valid[key].isin(label_counts.index[label_counts.gt(1)])])
    audit.loc[sorted(conflicts), "status"] = "label_conflict"

    candidates = audit["status"].eq("candidate")
    in_v1 = audit["sha256"].isin(base_sha) | audit["pixel_hash"].isin(base_pixels)
    in_v2 = audit["sha256"].isin(v2_sha) | audit["pixel_hash"].isin(v2_pixels)
    audit.loc[candidates & in_v1, "status"] = "duplicate_v1"
    audit.loc[candidates & ~in_v1 & in_v2, "status"] = "duplicate_v2_used"

    for key in ("sha256", "pixel_hash"):
        remaining = audit.loc[audit["status"].eq("candidate")]
        duplicates = remaining.index[remaining.duplicated(key)]
        audit.loc[duplicates, "status"] = "duplicate_within_hf"

    hf_index, hf_hashes = defaultdict(set), {}
    near = set()
    for i, item in audit.loc[audit["status"].eq("candidate"), "phash"].items():
        value = int(item, 16)
        if nearby_hashes(base_index, base_hashes, value) or nearby_hashes(v2_index, v2_hashes, value):
            near.add(i)
        neighbors = nearby_hashes(hf_index, hf_hashes, value)
        if neighbors:
            near.update(neighbors)
            near.add(i)
        add_hash(hf_index, hf_hashes, value, i)
    audit.loc[sorted(near), "status"] = "near_duplicate_review"
    audit.loc[audit["status"].eq("candidate"), "status"] = "selected"
    return audit


EXPECTED_HF_MANIFEST = "ca72b50d7e53db67d59c1cdc02267fc7afc1502e723935a2340d31a471915407"
experiment_dir = Path("residual_27_final")
experiment_dir.mkdir(exist_ok=True)
previous_dir = Path("trashbusters_residual_experiment")

def existing_hf_fingerprint(frame):
    selected = frame.loc[frame["status"].eq("selected"), ["id", "class", "split", "sha256"]]
    manifest = selected.sort_values("id").to_csv(index=False, lineterminator="\n")
    return hashlib.sha256((HF_REVISION + manifest).encode()).hexdigest()


def load_existing_hf(folder):
    audit_path = folder / "trashbusters_audit.csv"
    if not audit_path.is_file():
        return None
    audit = pd.read_csv(audit_path, dtype={"sha256": str, "phash": str, "pixel_hash": str})
    required_columns = {"id", "class", "split", "status", "path", "phash", "sha256", "pixel_hash"}
    if not required_columns.issubset(audit.columns) or existing_hf_fingerprint(audit) != EXPECTED_HF_MANIFEST:
        return None
    selected = audit.loc[audit["status"].eq("selected")].copy()
    paths = [folder / "images" / (digest + ".img") for digest in selected["sha256"]]
    for path, digest in zip(paths, selected["sha256"]):
        if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != digest:
            return None
    selected["path"] = [str(path) for path in paths]
    return selected


rebuild_dir = experiment_dir / "recreated_trashbusters"
selected_hf = load_existing_hf(previous_dir)
if selected_hf is None:
    selected_hf = load_existing_hf(rebuild_dir)
if selected_hf is None:
    hf_dir = Path(snapshot_download(HF_SOURCE, repo_type="dataset", revision=HF_REVISION,
                                    allow_patterns=["data/*.parquet"], token=False))
    parquet_files = {part: sorted((hf_dir / "data").glob(f"{part}-*.parquet")) for part in SPLITS}
    if not all(parquet_files.values()):
        raise FileNotFoundError("TrashBusters parquet의 세 분할을 확인해 주세요.")
    image_cache = rebuild_dir / "images"
    image_cache.mkdir(parents=True, exist_ok=True)
    previous_audit = deduplicate_hf(gather_hf_rows(parquet_files, image_cache), extra_split)
    if existing_hf_fingerprint(previous_audit) != EXPECTED_HF_MANIFEST:
        raise ValueError("재구성한 TrashBusters 분할이 #27 기록과 다릅니다. 기존 audit CSV와 images 폴더를 확인하세요.")
    previous_audit.to_csv(rebuild_dir / "trashbusters_audit.csv", index=False)
    selected_hf = previous_audit.loc[previous_audit["status"].eq("selected")].copy()

hf_train, hf_val, hf_test = [selected_hf.loc[selected_hf["split"].eq(part)].copy()
                             for part in ("train", "validation", "test")]
assert [len(hf_train), len(hf_val), len(hf_test)] == [3421, 774, 777]
base_train = pd.concat([combined_train, hf_train], ignore_index=True)
assert len(base_train) == 17068
print(f"#27 기존 학습 {len(base_train):,}장 | TrashBusters 검증 {len(hf_val):,}장 · 시험 {len(hf_test):,}장")

## 3 최종 학습 조건과 가중치 선택

V1·V2의 13,647장으로 잔차 CNN을 20 epoch 학습했습니다. Adam 학습률은 0.001이었습니다.
20번째 epoch에서 시작해 0.0003과 0.0001로 각각 5 epoch를 비교했습니다. 원본·변형 검증 평균 Macro F1이 가장 높았던 0.0001의 25번째 epoch가 #17의 가중치입니다.
확보한 #17 가중치를 시작점으로 세 클래스의 TrashBusters 학습 3,421장을 보강해 학습 17,068장으로 5 epoch 이어 학습했습니다.
선택식은 Kaggle 원본·변형의 평균 Macro F1과 TrashBusters 검증 Macro F1을 1:1로 평균합니다.
Adam 0.0001, 배치 32, 입력 128×128, BatchNorm 적용, Dropout과 무작위 학습 증강 미적용의 5번째 epoch를 선택했습니다.

In [ ]:
training_record = {'lr': 0.0001, 'dropout': 0.0, 'augmentation': 'none', 'batchnorm': True, 'blocks': 4, 'width': 'standard', 'image_size': 128, 'phase': 'continue', 'epochs': 5, 'seed': 42, 'trial': '3beb37c0c7e0fb20', 'epoch': 5, 'train_loss': 0.20342465055412812, 'train_accuracy_pct': 93.64893367705648, 'original_accuracy_pct': 87.80058651026394, 'original_macro_f1_pct': 85.89033484458923, 'original_loss': 0.46055730696647396, 'distorted_accuracy_pct': 80.64516129032258, 'distorted_macro_f1_pct': 78.48626971244812, 'distorted_loss': 0.7106559473748081, 'mean_macro_f1_pct': 82.18830227851868, 'hf_accuracy_pct': 72.09302325581395, 'hf_macro_f1_pct': 73.40525984764099, 'hf_loss': 0.8187440160941092, 'selection_score_pct': 77.79678106307983}


split_counts = pd.DataFrame([
    ["Kaggle V1", len(original_train), len(original_val), len(original_test)],
    ["Kaggle V2", len(extra_train), len(extra_val), len(extra_test)],
    ["TrashBusters", len(hf_train), len(hf_val), len(hf_test)],
], columns=["source", "train", "validation", "test"])
display(split_counts)
display(pd.crosstab(selected_hf["class"], selected_hf["split"]))
display(pd.DataFrame([training_record]).round(4).T)

## 4 제출 가중치로 평가

현재 환경에서 같은 분할을 평가합니다. PyTorch·Pillow 버전과 장치에 따라 값이 조금 달라질 수 있습니다.
변형 시험은 원본 V1 시험 이미지에 고정 시드로 자르기·축소·재확대를 적용한 비교이며, 교수님 생성 이미지 평가는 아래 별도 기록을 참고하세요.
Macro F1과 Macro Recall은 각 시험셋에 정답 표본이 있는 클래스를 평균합니다. 모델 출력은 모든 경우에 12개입니다.

In [ ]:
import random
from torch.nn import functional as F

JOINT_SCENARIO = "crop_low_res"
BATCH_SIZE = 32
VALIDATION_SEED = 723


def distort_image(image, scenario, rng):
    if scenario == "original":
        return image
    low_width = rng.randint(118, 183)

    # 원래 물체가 남도록 가로·세로 가장자리만 작게 자릅니다.
    if scenario in ("crop", JOINT_SCENARIO):
        width, height = image.size
        retained = rng.uniform(0.82, 0.94)
        crop_w = max(1, round(width * retained))
        crop_h = max(1, round(height * retained))
        left = rng.randint(0, width - crop_w)
        top = rng.randint(0, height - crop_h)
        image = image.crop((left, top, left + crop_w, top + crop_h))

    # 확대 전 정보량을 제한합니다. 색조는 그대로 유지합니다.
    if scenario in ("low_res", JOINT_SCENARIO):
        image = image.resize((low_width, 127), Image.Resampling.BILINEAR)
        image = image.resize((224, 224), Image.Resampling.BICUBIC)
    return image


class StressDataset(GarbageDataset):
    def __init__(self, frame, scenario, seed, augment_probability=0.0):
        super().__init__(frame, transform)
        self.scenario = scenario
        self.seed = seed
        self.augment_probability = augment_probability
        self.train_rng = random.Random(seed)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        scenario = self.scenario
        if scenario == "training_mix":
            rng = self.train_rng
            scenario = JOINT_SCENARIO if rng.random() < self.augment_probability else "original"
        else:
            rng = random.Random(self.seed + idx)

        with Image.open(row["path"]) as img:
            image = distort_image(img.convert("RGB"), scenario, rng)
            sample = self.transform(image)
        return sample, class_to_idx[row["class"]]


def evaluate_validation(net, loader):
    net.eval()
    size = len(classes)
    confusion = torch.zeros((size, size), dtype=torch.int64)
    total_loss = 0.0
    with torch.inference_mode():
        for x, y in loader:
            logits = net(x.to(device))
            total_loss += F.cross_entropy(logits, y.to(device), reduction="sum").item()
            positions = size * y + logits.argmax(1).cpu()
            confusion += torch.bincount(positions, minlength=size * size).reshape(size, size)

    support = confusion.sum(dim=1)
    correct = confusion.diag()
    total = support.sum().item()
    assert total == len(loader.dataset) and total > 0
    recall = correct / support.clamp_min(1)
    f1 = 2 * correct / (support + confusion.sum(dim=0)).clamp_min(1)
    return {
        "images": total,
        "correct": correct.sum().item(),
        "accuracy": correct.sum().item() / total,
        "macro_f1": f1[support > 0].mean().item(),
        "macro_recall": recall[support > 0].mean().item(),
        "loss": total_loss / total,
        "recall": recall.tolist(),
        "confusion": confusion.numpy(),
    }

def evaluation_loaders(frame, seed):
    return {
        scenario: DataLoader(StressDataset(frame, scenario, seed), batch_size=BATCH_SIZE)
        for scenario in ("original", JOINT_SCENARIO)
    }







import matplotlib.pyplot as plt
import numpy as np

test_cases = evaluation_loaders(original_test, VALIDATION_SEED)
loaders = {
    "V1 원본 시험": test_cases["original"],
    "V1 변형 시험": test_cases[JOINT_SCENARIO],
    "V2 보류 시험": DataLoader(GarbageDataset(extra_test, transform), batch_size=BATCH_SIZE),
    "TrashBusters 보류 시험": DataLoader(GarbageDataset(hf_test, transform), batch_size=BATCH_SIZE),
}
evaluated = {name: evaluate_validation(final_model, loader) for name, loader in loaders.items()}
comparison = pd.DataFrame([
    {"dataset": name, "images": row["images"], "correct": row["correct"],
     "accuracy_pct": row["accuracy"] * 100, "loss": row["loss"], "macro_f1_pct": row["macro_f1"] * 100,
     "macro_recall_pct": row["macro_recall"] * 100} for name, row in evaluated.items()
])
comparison.to_csv(experiment_dir / "test_comparison.csv", index=False)
display(comparison.round(4))

conf = evaluated["V1 원본 시험"]["confusion"]
pd.DataFrame(conf, index=classes, columns=classes).to_csv(experiment_dir / "v1_confusion.csv")
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(conf / np.maximum(conf.sum(axis=1, keepdims=True), 1), cmap="Blues", vmin=0, vmax=1)
ax.set(xlabel="Predicted class", ylabel="True class", title="V1 original test · confusion counts")
ax.set_xticks(range(len(classes)), classes, rotation=55, ha="right")
ax.set_yticks(range(len(classes)), classes)
for i, row in enumerate(conf):
    for j, count in enumerate(row):
        ax.text(j, i, str(count), ha="center", va="center", color="white" if i == j else "#20332d", fontsize=8)
plt.tight_layout()
plt.show()

## 5 교수님 평가 사이트 기록

제출 #27의 평가 결과를 옮겼습니다. 60장은 클래스별 5장으로 구성됐으며 모두 AI 생성 이미지입니다.
콜라주 42장은 낮은 해상도에서 잘라 확대했으므로, 전체 점수와 함께 독립 생성 18장 결과를 봅니다.
보강 후보를 비교할 때 교수님 평가 결과를 참고했습니다. 이 기록을 실제 배출 환경에 대한 성능 추정으로 확대하지 않습니다.

In [ ]:
official = [{'set': '전체', 'images': 60, 'correct': 37, 'loss': 1.8831}, {'set': '콜라주 확대', 'images': 42, 'correct': 23, 'loss': 2.1689}, {'set': '독립 생성', 'images': 18, 'correct': 14, 'loss': 1.2163}]
official_by_class = {'battery': 2, 'biological': 5, 'brown-glass': 2, 'cardboard': 5, 'clothes': 2, 'green-glass': 4, 'metal': 2, 'paper': 2, 'plastic': 5, 'shoes': 4, 'trash': 1, 'white-glass': 3}


report = pd.DataFrame(official)
report["accuracy_pct"] = 100 * report["correct"] / report["images"]
display(report.round(4))
display(pd.Series(official_by_class, name="correct_out_of_5").to_frame())

## 6 선택 실행 학습 재현과 하이퍼파라미터 탐색

아래 설정의 `RUN_REPLAY`를 `True`로 하면 보관된 #17 가중치에서 5 epoch 동안 원래의 이어 학습을 재현합니다.
`RUN_FROM_SCRATCH`는 잔차 CNN 무작위 초기화에서 시작해 V1·V2 20 epoch, 낮은 학습률 비교, TrashBusters 추가 학습을 실행합니다.
`RUN_HYPERPARAMETER_SEARCH`는 이어 학습의 학습률·Dropout·증강을 순차 비교하는 별도 실험입니다.
출력은 실험 폴더에 저장되며 기본 평가에 쓰인 제출 가중치는 별도로 유지됩니다.
초기 3블록 CNN부터 WideCNN으로 바꾼 학습률·BatchNorm·규제·블록 수·폭·입력 크기의 비교표는 보고서와 [탐색 저장소](https://github.com/njw257/bigdata)를 참고하세요.

In [ ]:
#@title #17 가중치 (선택 실행용)
EMBEDDED_PRE_TRASH_WEIGHTS = (
    "..."
)
EXPECTED_PRE_TRASH_SHA256 = "81ef998412c9803de5797b9a9f1018400d29c97769a55bf444f7ddbf014f1cc2"

In [ ]:
def comparison_scores(net, loaders):
    original = evaluate_validation(net, loaders["original"])
    shifted = evaluate_validation(net, loaders[JOINT_SCENARIO])
    return {
        "original_accuracy_pct": 100 * original["accuracy"],
        "original_macro_f1_pct": 100 * original["macro_f1"],
        "original_loss": original["loss"],
        "distorted_accuracy_pct": 100 * shifted["accuracy"],
        "distorted_macro_f1_pct": 100 * shifted["macro_f1"],
        "distorted_loss": shifted["loss"],
        "mean_macro_f1_pct": 50 * (original["macro_f1"] + shifted["macro_f1"]),
    }




import json
from dataclasses import asdict, dataclass, replace


@dataclass(frozen=True)
class Trial:
    lr: float = 0.0001
    dropout: float = 0.0
    augmentation: str = "none"
    batchnorm: bool = True
    blocks: int = 4
    width: str = "standard"
    image_size: int = 128


WIDTHS = {
    "narrow": (24, 48, 96, 144),
    "standard": (32, 64, 128, 192),
    "wide": (40, 80, 160, 224),
}
SEARCH_SEED = 42
search_dir = experiment_dir / "hyperparameter_search"
search_dir.mkdir(exist_ok=True)
data_fingerprint = EXPECTED_HF_MANIFEST
expanded_train = base_train


def load_pre_trash_state():
    payload = zlib.decompress(base64.b64decode(EMBEDDED_PRE_TRASH_WEIGHTS))
    if hashlib.sha256(payload).hexdigest() != EXPECTED_PRE_TRASH_SHA256:
        raise ValueError("#17 시작 가중치 SHA-256이 일치하지 않습니다.")
    return torch.load(io.BytesIO(payload), map_location="cpu", weights_only=True)


class SearchBlock(ResidualBlock):
    def __init__(self, in_channels, out_channels, batchnorm):
        super().__init__(in_channels, out_channels)
        if not batchnorm:
            self.convs = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, padding=1), nn.ReLU(),
                nn.Conv2d(out_channels, out_channels, 3, padding=1),
            )
            if in_channels != out_channels:
                self.shortcut = nn.Conv2d(in_channels, out_channels, 1)


class SearchResidualCNN(nn.Module):
    def __init__(self, config):
        super().__init__()
        outputs = WIDTHS[config.width][:config.blocks]
        inputs = (3,) + outputs[:-1]
        self.features = nn.Sequential(*[
            SearchBlock(cin, cout, config.batchnorm) for cin, cout in zip(inputs, outputs)
        ])
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(outputs[-1], len(classes))
        )
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.classifier[1](self.classifier[0](self.features(x)))
        return self.classifier[2](self.dropout(x))


def image_transform(size, augmentation="none"):
    steps = [transforms.Resize((size, size))]
    if augmentation in ("geometry", "geometry_lighting"):
        steps.extend([
            transforms.RandomHorizontalFlip(),
            transforms.RandomAffine(10, translate=(0.05, 0.05), scale=(0.95, 1.05),
                                    interpolation=transforms.InterpolationMode.BILINEAR),
        ])
    if augmentation == "geometry_lighting":
        steps.append(transforms.ColorJitter(brightness=0.15, contrast=0.15))
    steps.extend([transforms.ToTensor(), transforms.Normalize([0.5] * 3, [0.5] * 3)])
    return transforms.Compose(steps)


def loaders_for_size(frame, size, seed):
    loaders = evaluation_loaders(frame, seed)
    for loader in loaders.values():
        loader.dataset.transform = image_transform(size)
    return loaders


def train_trial(config, phase, epochs, *, kaggle_only=False, starting_weights=None):
    settings = {**asdict(config), "phase": phase, "epochs": epochs, "seed": SEARCH_SEED,
                "initial_checkpoint": EXPECTED_PRE_TRASH_SHA256, "data_fingerprint": data_fingerprint}
    if kaggle_only:
        settings["data_fingerprint"] = hashlib.sha256((EXPECTED_DATA_SHA256 + EXPECTED_V2_SPLIT_SHA256).encode()).hexdigest()
    if starting_weights is not None:
        settings["initial_checkpoint"] = hashlib.sha256(Path(starting_weights).read_bytes()).hexdigest()
    uid = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()[:16]
    trial_dir = search_dir / uid
    trial_dir.mkdir(exist_ok=True)
    weights = trial_dir / "weights.pt"
    summary = trial_dir / "best.json"
    if summary.is_file() and weights.is_file():
        print(f"{uid} 완료 결과 재사용")
        return json.loads(summary.read_text(encoding="utf-8"))

    torch.manual_seed(SEARCH_SEED)
    net = SearchResidualCNN(config).to(device)
    if starting_weights is not None:
        net.load_state_dict(torch.load(starting_weights, map_location="cpu", weights_only=True))
    elif phase == "continue":
        net.load_state_dict(load_pre_trash_state())
    optimizer = torch.optim.Adam(net.parameters(), lr=config.lr)
    frame = combined_train if kaggle_only else expanded_train
    dataset = GarbageDataset(frame, image_transform(config.image_size, config.augmentation))
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        generator=torch.Generator().manual_seed(SEARCH_SEED))
    val_loaders = loaders_for_size(combined_val, config.image_size, VALIDATION_SEED)
    history, best = [], None
    print(f"{uid} | {phase} | {config}")

    for epoch in range(1, epochs + 1):
        net.train()
        loss_sum, correct = 0.0, 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = net(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()

        if kaggle_only:
            scores = comparison_scores(net, val_loaders)
            scores["selection_score_pct"] = scores["mean_macro_f1_pct"]
        else:
            scores = data_validation_scores(net, config.image_size, val_loaders)

        row = {
            **settings, "trial": uid, "epoch": epoch,
            "train_loss": loss_sum / len(dataset), "train_accuracy_pct": 100 * correct / len(dataset),
            "weights_path": str(weights), **scores,
        }
        history.append(row)
        if best is None or row["selection_score_pct"] > best["selection_score_pct"]:
            best = row
            temporary = weights.with_suffix(".tmp")
            torch.save({key: value.detach().cpu() for key, value in net.state_dict().items()}, temporary)
            temporary.replace(weights)
        pd.DataFrame(history).to_csv(trial_dir / "history.csv", index=False)
        print(f"{uid} {epoch}/{epochs} | 원본 {row['original_accuracy_pct']:.2f}% | "
              f"변형 {row['distorted_accuracy_pct']:.2f}% | "
              f"선택 점수 {row['selection_score_pct']:.2f}%")

    temporary = summary.with_suffix(".tmp")
    temporary.write_text(json.dumps(best, ensure_ascii=False), encoding="utf-8")
    temporary.replace(summary)
    return best


def compare(stage, candidates, phase, epochs):
    configs = list(dict.fromkeys(candidates))
    rows = [train_trial(config, phase, epochs) for config in configs]
    table = pd.DataFrame(rows).sort_values("selection_score_pct", ascending=False, kind="stable")
    table.to_csv(search_dir / f"{stage}.csv", index=False)
    print(f"비교 결과 · {stage}")
    display(table.drop(columns=["initial_checkpoint", "seed", "data_fingerprint"]).round(3))
    winner = max(range(len(rows)), key=lambda i: rows[i]["selection_score_pct"])
    return configs[winner], rows[winner]

def data_validation_scores(net, size, val_loaders=None):
    if val_loaders is None:
        val_loaders = loaders_for_size(combined_val, size, VALIDATION_SEED)
    old_metrics = comparison_scores(net, val_loaders)
    hf_loader = DataLoader(GarbageDataset(hf_val, image_transform(size)), batch_size=BATCH_SIZE)
    fresh = evaluate_validation(net, hf_loader)
    hf_macro_f1_pct = 100 * fresh["macro_f1"]
    return {
        **old_metrics,
        "hf_accuracy_pct": 100 * fresh["accuracy"],
        "hf_macro_f1_pct": hf_macro_f1_pct,
        "hf_loss": fresh["loss"],
        "selection_score_pct": (old_metrics["mean_macro_f1_pct"] + hf_macro_f1_pct) / 2,
    }


def save_validation_comparison(candidates, filename):
    table = pd.DataFrame.from_dict(candidates, orient="index").sort_values(
        "selection_score_pct", ascending=False, kind="stable"
    )
    table.to_csv(search_dir / filename)
    display(table.drop(columns=["initial_checkpoint", "seed", "data_fingerprint"], errors="ignore").round(3))
    return table

In [ ]:
RUN_REPLAY = False
RUN_FROM_SCRATCH = False
RUN_HYPERPARAMETER_SEARCH = False
reference = Trial()

if RUN_REPLAY:
    replay_result = train_trial(reference, "continue", 5)
    display(pd.DataFrame([replay_result]).round(4))

if RUN_FROM_SCRATCH:
    fresh = train_trial(replace(reference, lr=0.001), "fresh", 20, kaggle_only=True)
    lowered = [train_trial(replace(reference, lr=lr), "kaggle_followup", 5, kaggle_only=True,
                           starting_weights=fresh["weights_path"]) for lr in (0.0003, 0.0001)]
    start = max(lowered, key=lambda row: row["selection_score_pct"])
    replay_from_scratch = train_trial(reference, "trashbusters_followup", 5, starting_weights=start["weights_path"])
    display(pd.DataFrame([fresh, *lowered, replay_from_scratch]).round(4))

if RUN_HYPERPARAMETER_SEARCH:
    best, _ = compare("01_learning_rate", [replace(reference, lr=lr) for lr in (0.00003, 0.0001, 0.0003)],
                      "continue", 3)
    best, _ = compare("02_dropout", [replace(best, dropout=p) for p in (0.0, 0.1, 0.2, 0.3, 0.4, 0.5)],
                      "continue", 3)
    best, _ = compare("03_augmentation", [replace(best, augmentation=a) for a in
                      ("none", "geometry", "geometry_lighting")], "continue", 3)
    print(best)

## 7 선택 실행 평가용 파일 내려받기

이미 제출 폴더에 `model.py`와 `weights.pt`가 들어 있습니다. Colab에서 다시 받으려면 `DOWNLOAD_EVALUATOR`를 `True`로 바꾸세요.

In [ ]:
EVALUATOR_SOURCE = '"""쓰레기 분류 챌린지 평가용 잔차 CNN과 입력 전처리."""\n\nimport torch\nfrom torch import nn\nfrom torchvision import transforms\n\n\nCLASSES = [\n    "battery",\n    "biological",\n    "brown-glass",\n    "cardboard",\n    "clothes",\n    "green-glass",\n    "metal",\n    "paper",\n    "plastic",\n    "shoes",\n    "trash",\n    "white-glass",\n]\n\n\nclass ResidualBlock(nn.Module):\n    def __init__(self, in_channels, out_channels):\n        super().__init__()\n        self.convs = nn.Sequential(\n            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),\n            nn.BatchNorm2d(out_channels),\n            nn.ReLU(),\n            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),\n            nn.BatchNorm2d(out_channels),\n        )\n        self.shortcut = nn.Identity()\n        if in_channels != out_channels:\n            self.shortcut = nn.Sequential(\n                nn.Conv2d(in_channels, out_channels, 1, bias=False),\n                nn.BatchNorm2d(out_channels),\n            )\n        self.activation = nn.ReLU()\n        self.pool = nn.MaxPool2d(2)\n\n    def forward(self, x):\n        return self.pool(self.activation(self.convs(x) + self.shortcut(x)))\n\n\nclass ResidualCNN(nn.Module):\n    def __init__(self, num_classes):\n        super().__init__()\n        self.features = nn.Sequential(\n            ResidualBlock(3, 32),\n            ResidualBlock(32, 64),\n            ResidualBlock(64, 128),\n            ResidualBlock(128, 192),\n        )\n        self.classifier = nn.Sequential(\n            nn.AdaptiveAvgPool2d(1),\n            nn.Flatten(),\n            nn.Linear(192, num_classes),\n        )\n\n    def forward(self, x):\n        return self.classifier(self.features(x))\n\n\ndef load_model(weights_path, device):\n    model = ResidualCNN(len(CLASSES)).to(device)\n    state = torch.load(weights_path, map_location="cpu", weights_only=True)\n    model.load_state_dict(state)\n    return model.eval()\n\n\ndef get_transform():\n    return transforms.Compose([\n        transforms.Resize((128, 128)),\n        transforms.ToTensor(),\n        transforms.Normalize([0.5] * 3, [0.5] * 3)\n    ])\n'


DOWNLOAD_EVALUATOR = False
if DOWNLOAD_EVALUATOR:
    from google.colab import files

    evaluator_file = experiment_dir / "model.py"
    weights_file = experiment_dir / "weights.pt"
    evaluator_file.write_text(EVALUATOR_SOURCE, encoding="utf-8")
    weights_file.write_bytes(residual_bytes)
    files.download(str(evaluator_file))
    files.download(str(weights_file))

## 8 선택 실행 외부 이미지 한 장 분류

파일을 올리려면 `RUN_IMAGE_PREDICTION`을 `True`로 설정하세요. 제출 가중치로 상위 3개 클래스를 표시합니다.

In [ ]:
RUN_IMAGE_PREDICTION = False
if RUN_IMAGE_PREDICTION:
    from google.colab import files

    uploaded = files.upload()
    for filename, payload in uploaded.items():
        with Image.open(io.BytesIO(payload)) as image:
            pixels = transform(image.convert("RGB")).unsqueeze(0).to(device)
        with torch.inference_mode():
            confidence, indices = final_model(pixels).softmax(dim=1)[0].topk(3)
        print(filename)
        for score, index in zip(confidence.tolist(), indices.tolist()):
            print(f"{classes[index]}: {score:.2%}")